In [1]:
from google.colab import files
uploaded = files.upload()


Saving rus.txt to rus.txt


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random


In [26]:
def clean_text(text):
    return text.lower()

def load_data(path, max_samples=20000):
    pairs = []
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            parts = line.strip().split('\t', 2)
            if len(parts) >= 2:
                eng = parts[0]
                rus = parts[1]
                rus = "<sos> " + clean_text(rus) + " <eos>"
                eng = clean_text(eng)
                pairs.append((eng, rus))
            else:
                print(f"Skipping malformed line (expected at least two parts): {line.strip()}")
    return pairs

data = load_data("rus.txt")

print("Sample:", data[0])

Sample: ('go.', '<sos> марш! <eos>')


In [27]:
class Vocab:
    def __init__(self):
        self.word2idx = {"<pad>":0, "<sos>":1, "<eos>":2}
        self.idx2word = {0:"<pad>", 1:"<sos>", 2:"<eos>"}
        self.idx = 3

    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def encode(self, sentence):
        return [self.word2idx[w] for w in sentence.split()]

eng_vocab = Vocab()
rus_vocab = Vocab()

for eng, rus in data:
    eng_vocab.add_sentence(eng)
    rus_vocab.add_sentence(rus)

print("English vocab size:", len(eng_vocab.word2idx))
print("Russian vocab size:", len(rus_vocab.word2idx))


English vocab size: 3930
Russian vocab size: 9924


In [28]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        eng, rus = self.data[idx]
        return torch.tensor(eng_vocab.encode(eng)), torch.tensor(rus_vocab.encode(rus))

def collate_fn(batch):
    eng_batch, rus_batch = zip(*batch)

    eng_batch = nn.utils.rnn.pad_sequence(eng_batch, padding_value=0)
    rus_batch = nn.utils.rnn.pad_sequence(rus_batch, padding_value=0)

    return eng_batch, rus_batch

dataset = TranslationDataset(data)
loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [29]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim=256, hid_dim=1024):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


In [30]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim=256, hid_dim=1024):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim)
        self.fc = nn.Linear(hid_dim, output_dim)

    def forward(self, x, hidden, cell):
        x = x.unsqueeze(0)
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden, cell


In [31]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = len(rus_vocab.word2idx)

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(device)

        hidden, cell = self.encoder(src)

        input = trg[0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = trg[t] if teacher_force else top1

        return outputs


In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(len(eng_vocab.word2idx)).to(device)
decoder = Decoder(len(rus_vocab.word2idx)).to(device)

model = Seq2Seq(encoder, decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)


In [33]:
def train(model, loader):
    model.train()
    total_loss = 0

    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        output = output[1:].reshape(-1, output.shape[-1])
        trg = trg[1:].reshape(-1)

        loss = criterion(output, trg)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


In [34]:
for epoch in range(20):
    loss = train(model, loader)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


Epoch 1, Loss: 4.2820
Epoch 2, Loss: 2.6430
Epoch 3, Loss: 1.6358
Epoch 4, Loss: 1.1696
Epoch 5, Loss: 0.9780
Epoch 6, Loss: 0.8694
Epoch 7, Loss: 0.8165
Epoch 8, Loss: 0.7718
Epoch 9, Loss: 0.7431
Epoch 10, Loss: 0.7140
Epoch 11, Loss: 0.6998
Epoch 12, Loss: 0.6708
Epoch 13, Loss: 0.6602
Epoch 14, Loss: 0.6471
Epoch 15, Loss: 0.6312
Epoch 16, Loss: 0.6179
Epoch 17, Loss: 0.6041
Epoch 18, Loss: 0.5951
Epoch 19, Loss: 0.5823
Epoch 20, Loss: 0.5800


In [37]:
def translate(sentence):
    model.eval()

    tokens = eng_vocab.encode(clean_text(sentence))
    src = torch.tensor(tokens).unsqueeze(1).to(device)

    hidden, cell = model.encoder(src)

    input = torch.tensor([rus_vocab.word2idx["<sos>"]]).to(device)

    result = []

    for _ in range(20):
        output, hidden, cell = model.decoder(input, hidden, cell)
        top1 = output.argmax(1).item()

        if top1 == rus_vocab.word2idx["<eos>"]:
            break

        result.append(rus_vocab.idx2word[top1])
        input = torch.tensor([top1]).to(device)

    return " ".join(result)


In [39]:
print(translate("how are you"))


как вы
